# Module 3.2 - Re-ranking and Hybrid Search

**Reference:** [`02-reranking-and-hybrid-search.md`](./02-reranking-and-hybrid-search.md)

## What you'll do in this notebook

- Use an LLM as a cheap re-ranker on top of vector search.
- Add a BM25 keyword-search index alongside the vector store.
- Combine both with an `alpha` knob and see how dense/sparse trade off.

> Production systems often use a small cross-encoder model (e.g. `cross-encoder/ms-marco-MiniLM-L-6-v2`) for re-ranking. That requires `sentence-transformers` and pulls in PyTorch (~2 GB). In this notebook we use an LLM-based re-ranker instead, which needs no extra install and is more than enough to learn the concept.

## Setup

In [ ]:
import os
import shutil
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from rank_bm25 import BM25Okapi

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
EMBED_MODEL = "text-embedding-3-small"
CHROMA_PATH = "./chroma_store_m3_2"

if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

chroma = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma.create_collection("rerank_demo")

def embed_batch(texts):
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in resp.data]

DOCS = [
    "Exception handling in Python is done with try/except/finally blocks.",
    "Python decorators wrap a function to extend its behaviour.",
    "A Python list comprehension is a concise way to build a list from an iterable.",
    "Error codes in the XR-7821 product line start with the prefix 'XR7-'.",
    "The XR-7821 device supports firmware rollback via its recovery mode.",
    "Requests is a popular Python HTTP client library.",
    "The httpx library in Python supports both sync and async HTTP calls.",
    "Context managers free resources deterministically when a block exits.",
]

ids = [f"doc_{i}" for i in range(len(DOCS))]
collection.add(ids=ids, embeddings=embed_batch(DOCS), documents=DOCS, metadatas=[{"source": f"{i}"} for i in range(len(DOCS))])
print(f"OK: {collection.count()} documents indexed")

## Exercise 1 - Vector search baseline

Start by running a plain vector search so you have something to compare against.

In [ ]:
def vector_search(query: str, top_k: int = 5) -> list[dict]:
    emb = embed_batch([query])[0]
    res = collection.query(query_embeddings=[emb], n_results=top_k)
    return [
        {"id": i, "text": d, "distance": dist}
        for i, d, dist in zip(res["ids"][0], res["documents"][0], res["distances"][0])
    ]

for hit in vector_search("How do I catch errors in my Python code?", top_k=5):
    print(f"[{hit['distance']:.3f}] ({hit['id']}) {hit['text']}")

## Exercise 2 - LLM-based re-ranking

Retrieve a wider candidate set (say 6), then ask the LLM to score each candidate against the query on a 0-10 scale. Resort the candidates by the score.

In [ ]:
RERANK_SYSTEM = (
    "You are a search relevance judge. Given a query and a candidate document, "
    "rate how directly the document answers the query from 0 (completely irrelevant) "
    "to 10 (the perfect answer). Reply with the number only, no words."
)

def score_with_llm(query: str, document: str) -> float:
    # TODO:
    # 1. Call client.chat.completions.create with:
    #    - system=RERANK_SYSTEM
    #    - user=f"Query: {query}\nDocument: {document}\nScore:"
    #    - temperature=0, max_tokens=4
    # 2. Parse the reply as a float; on parse error return 0.0.
    raise NotImplementedError

def retrieve_and_rerank(query: str, initial_k: int = 6, final_k: int = 3) -> list[dict]:
    candidates = vector_search(query, top_k=initial_k)
    # TODO: call score_with_llm on each candidate, attach "rerank_score",
    # sort desc by rerank_score, and return the top final_k.
    raise NotImplementedError

query = "How do I catch errors in my Python code?"
print("-- baseline vector search --")
for hit in vector_search(query, top_k=3):
    print(f"[{hit['distance']:.3f}] ({hit['id']}) {hit['text']}")

print("\n-- after LLM rerank --")
for hit in retrieve_and_rerank(query):
    print(f"[rerank={hit['rerank_score']:.1f}] ({hit['id']}) {hit['text']}")

## Exercise 3 - Sparse retrieval with BM25

Dense search understands meaning but can miss exact-string matches (like product codes). BM25 is the classical counterpoint: it scores by keyword overlap with frequency weighting.

The `rank-bm25` package is in `requirements.txt` and should already be installed.

In [ ]:
def tokenize(text: str) -> list[str]:
    return text.lower().split()

tokenised_docs = [tokenize(d) for d in DOCS]
bm25 = BM25Okapi(tokenised_docs)

def bm25_search(query: str, top_k: int = 5) -> list[dict]:
    # TODO:
    # 1. Tokenise the query.
    # 2. Use bm25.get_scores(tokens) to get a score for every document.
    # 3. Pair each score with its doc id and text, sort desc by score, return top_k.
    raise NotImplementedError

print("-- BM25 on a product code query --")
for hit in bm25_search("XR-7821 error codes", top_k=3):
    print(f"[bm25={hit['score']:.2f}] ({hit['id']}) {hit['text']}")

print("\n-- BM25 on a semantic query --")
for hit in bm25_search("catch errors in code", top_k=3):
    print(f"[bm25={hit['score']:.2f}] ({hit['id']}) {hit['text']}")

You should see BM25 nail the product-code query (exact keyword match) but struggle on the semantic "catch errors" query. That's exactly the gap hybrid search closes.

## Exercise 4 - Hybrid search

Run both searches and combine them with a weighted sum. `alpha=1.0` is pure dense, `alpha=0.0` is pure sparse, `alpha=0.5` is a 50-50 mix.

In [ ]:
def normalise(values):
    values = np.asarray(values, dtype=float)
    if values.max() == values.min():
        return np.zeros_like(values)
    return (values - values.min()) / (values.max() - values.min())

def hybrid_search(query: str, top_k: int = 5, alpha: float = 0.5) -> list[dict]:
    """Combine normalised dense similarity and BM25 with weight alpha."""
    # Dense scores: convert distance -> similarity = 1 - distance.
    dense_hits = vector_search(query, top_k=len(DOCS))
    dense_sim_by_id = {h["id"]: 1 - h["distance"] for h in dense_hits}

    # Sparse scores: BM25 over all docs in the indexed order.
    bm25_scores = bm25.get_scores(tokenize(query))

    dense_norm = normalise([dense_sim_by_id[i] for i in ids])
    sparse_norm = normalise(bm25_scores)

    # TODO: build a list of dicts {id, text, score} where
    #   score = alpha * dense_norm[idx] + (1 - alpha) * sparse_norm[idx]
    # then sort desc by score and return the top_k.
    raise NotImplementedError

query_semantic = "catching errors in Python programs"
query_code = "XR-7821 error codes"

for q in [query_semantic, query_code]:
    print(f"\nquery: {q!r}")
    for alpha in (0.0, 0.3, 0.5, 0.7, 1.0):
        top = hybrid_search(q, top_k=1, alpha=alpha)[0]
        print(f"  alpha={alpha:.1f} -> ({top['id']}) {top['text']}")

**Reflect:**

- What value of `alpha` wins for each query? Is there a single `alpha` that's best for both?
- How would you decide alpha in production? (Hint: look at the query for identifiers, numbers, or quoted strings.)

## What's next

Sometimes the right answer isn't a passage at all — it's a fact tied into a web of other facts ("who works at which company, built which framework, written in which language"). `03-knowledge-graphs.ipynb` introduces a complementary representation: the knowledge graph.